In [1]:
import numpy as np
from transformers import BertModel, BertTokenizer
from tqdm import tqdm


d:\DESIGN\ANACONDA\envs\bertopic_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Imports

In [ ]:
with open("../data/02_cleaned/cleaned_responses_397.txt", "r", encoding="utf-8") as file:
    sentences = file.readlines()

print("Number of sentences:", len(sentences))
print("First sentence preview:", sentences[0])


Number of sentences:  397
First sentence preview:  买错衣服尺码的时候 换货很麻烦



In [4]:
model_name = "bert-base-chinese"

model = BertModel.from_pretrained(model_name)

tokenizer = BertTokenizer.from_pretrained(model_name)


Some weights of the model checkpoint at bert-base-chinese were not used when initializing BertModel: ['cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [5]:
import torch
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

model.eval()

batch_size = 16
data_loader = DataLoader(sentences, batch_size=batch_size)
for batch in data_loader:
    print(len(batch), batch)


16 ['买错衣服尺码的时候 换货很麻烦\n', '衣柜满了但感觉没有衣服穿\n', '不能退换货\n', '尺寸不对\n', '衣柜整理\n', '没有需求有衣服穿就行\n', '好看的衣服不好洗，好洗的衣服太平庸，很多材质费时费力巴不得雇个佣人帮自己打理\n', '衣柜满了但是没有衣服穿\n', '换季时整理衣柜的崩溃瞬间\n', '换季整理麻烦\n', '衣橱太小装不了全部衣服\n', '衣柜满了，但没有衣服穿\n', '宿舍衣橱太小，各种季节的衣服堆满了衣橱，感觉有些衣服穿了一次就不想穿了\n', '学校衣柜太小，又要放冬天的床上用品，还要放衣服，这样衣服都挂不起来只能堆着。\n', '衣柜满了没衣服穿，衣服种类多，不易共同存放\n', '换季时没有衣服可穿，要么太厚要么太薄；买的新衣服刚开始很喜欢，刚穿一两次就觉得不喜欢，后来就下也不想穿了\n']
16 ['购买衣服价格和质量挂钩，要长期穿既要考虑质量又要考虑价格，又要看适不适合自己穿出去，质量好的衣服清洗也很麻烦，有时会送到洗衣店清洗，收纳方面就是夏天衣服放在一起，冬天衣服放在一起\n', '衣服脏了不好清洗，衣服多，不好整理，收纳空间小\n', '我曾在网上买了一件很喜欢的连衣裙，价格不便宜，满心期待地收到后试穿，发现版型和模特图差距很大，根本不适合自己。想退货时，却发现商家以“已试穿”为由拒绝了，沟通多次都无果。看着那件穿不了又退不掉的裙子，既心疼钱又觉得很无奈，感觉自己像吃了个闷亏，之后好长一段时间都不敢轻易网购衣服了。\n', '衣服很多但是总是不知道要穿什么\n', '随着不同季节 不同时间 穿搭需求一直在变化 会出现购买的衣服只穿了几次就放到衣橱的情况 同时带来衣柜满了但没衣服穿的烦恼 会觉得处理起来很麻烦 扔了也不是不扔也不是\n', '衣服样式太多不好整理心情烦躁，衣柜满了但是没衣服穿选择困难症\n', '宿舍衣橱小，装不下衣服\n', '衣柜的储存位置不够，不能好好分类衣服种类，只有上面一个空间，下面挂衣服，不好分类\n', '衣服不合适，只能放着，但是占空间。衣柜不能分类，能分类又太小了。不好清理，木质衣柜味道大。常用衣服跟衣服距离太小，经常找不到想要的衣服，拿衣服容易把其他的衣服一起带出来。\n', '出去买衣服的时候样式很多，但是不会选择搭配，单件效果不错，但是买回去穿成整套效果

In [ ]:
cls_embeddings = []

for batch_sentences in tqdm(data_loader):

    inputs = tokenizer(batch_sentences, padding=True, truncation=True, return_tensors="pt", max_length=512)

    inputs.to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    cls_embeddings.append(outputs.last_hidden_state[:, 0].cpu().numpy())

    print('NumPy format', type(outputs.last_hidden_state[:, 0].cpu().numpy()), outputs.last_hidden_state[:, 0].cpu().numpy().shape)

print('Number of batches:', len(cls_embeddings))
cls_embeddings_np = np.vstack(cls_embeddings)
print('Final embeddings:', type(cls_embeddings_np), cls_embeddings_np.shape)

output_file = "../data/02_bertopic_outputs/data_emb.npy"

np.save(output_file, cls_embeddings_np)
print("Embeddings saved to:", output_file)

embeddings = np.load(output_file)
print("Reloaded embeddings for verification:", type(embeddings), embeddings.shape)

  8%|▊         | 2/25 [00:00<00:01, 11.56it/s]

NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)


 20%|██        | 5/25 [00:00<00:01, 14.87it/s]

NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)


 44%|████▍     | 11/25 [00:00<00:00, 17.85it/s]

NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)


 68%|██████▊   | 17/25 [00:00<00:00, 21.73it/s]

NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)


100%|██████████| 25/25 [00:01<00:00, 21.18it/s]

NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (16, 768)
NumPy format <class 'numpy.ndarray'> (13, 768)
Number of batches: 25
Final embeddings: <class 'numpy.ndarray'> (397, 768)
Embeddings saved to:  data_emb.npy
Reloaded embeddings for verification: <class 'numpy.ndarray'> (397, 768)
